# Actividad — Análisis de Violaciones Invariante 1: `run-02-apt-1`

**Objetivo:** aplicar la metodología de recuperación de GUID estudiada en la sección 9
a un run distinto del visto en clase, descubriendo qué patrones se replican
y cuáles son nuevos.

**Archivos de trabajo:**
- `02_sysmon-run-02.csv` — CSV Sysmon del run 02
- Este notebook

**Tiempo estimado:** 60–75 minutos


## Setup

In [ ]:
import pandas as pd

NULL_GUID = '00000000-0000-0000-0000-000000000000'

CSV_PATH = '../dataset/run-02-apt-1/02_sysmon-run-02.csv'

df = pd.read_csv(CSV_PATH, low_memory=False)
df['ts'] = pd.to_datetime(df['timestamp'].where(df['timestamp'] > 0), unit='ms')

print(f"Filas: {len(df):,}   Columnas: {df.shape[1]}")
print(f"Rango temporal: {df['ts'].min()} → {df['ts'].max()}")


---
## Actividad 1 — Explorar las violaciones por k-par

Antes de analizar casos individuales, obtén una visión general:
¿cuántos eventos centinela hay en cada k-par? ¿cuántos PIDs únicos los generan?

La tabla de referencia de `run-01-apt-1` era:

| k | Centinelas | PIDs únicos |
|---|:----------:|:-----------:|
| 1 |     36     |     14      |
| 2 |    500     |     24      |
| 3 |      0     |      0      |
| 4 |      4     |      2      |
| ∑ |    540     |     —       |


In [ ]:
# Definición de los cuatro dominios de centinela
k1_sent = df[~df['EventID'].isin([8, 10]) & (df['ProcessGuid']       == NULL_GUID)]
k2_sent = df[ (df['EventID'] == 1)        & (df['ParentProcessGuid'] == NULL_GUID)]
k3_sent = df[ df['EventID'].isin([8, 10]) & (df['SourceProcessGUID'] == NULL_GUID)]
k4_sent = df[ df['EventID'].isin([8, 10]) & (df['TargetProcessGUID'] == NULL_GUID)]

print(f"k=1: {len(k1_sent):4d} centinelas  {k1_sent['ProcessId'].nunique():3d} PIDs únicos")
print(f"k=2: {len(k2_sent):4d} centinelas  {k2_sent['ParentProcessId'].nunique():3d} PIDs únicos")
print(f"k=3: {len(k3_sent):4d} centinelas  {k3_sent['SourceProcessId'].nunique():3d} PIDs únicos")
print(f"k=4: {len(k4_sent):4d} centinelas  {k4_sent['TargetProcessId'].nunique():3d} PIDs únicos")
print(f"∑  : {len(k1_sent)+len(k2_sent)+len(k3_sent)+len(k4_sent):4d}")


```{admonition} Pregunta
:class: tip
¿Qué diferencias observas respecto a `run-01-apt-1`? ¿Qué k-par aparece ahora
que no existía? ¿Cuál desaparece casi por completo?
```


---
## Actividad 2 — Implementar `compute_gset`

El conjunto $\mathcal{G}(p, c)$ reúne todos los GUIDs **reales** (no centinela)
observados para el proceso $(p, c)$ a través de los cuatro k-pares:

$$
\mathcal{G}(p,\, c) = \left(\bigcup_{k=1}^{4} \mathcal{G}_k(p,\, c)\right) \setminus \{\emptyset\}
$$

Donde cada $\mathcal{G}_k$ busca en un subconjunto de eventos y una columna distinta:

| k | Columna GUID | Columna PID | Dominio de EIDs |
|---|-------------|------------|-----------------|
| 1 | `ProcessGuid` | `ProcessId` | EID ∉ {8, 10} |
| 2 | `ParentProcessGuid` | `ParentProcessId` | EID = 1 |
| 3 | `SourceProcessGUID` | `SourceProcessId` | EID ∈ {8, 10} |
| 4 | `TargetProcessGUID` | `TargetProcessId` | EID ∈ {8, 10} |

**Tu tarea:** completar la función con los k-pares que faltan.


In [ ]:
def compute_gset(df, pid, comp):
    """
    Devuelve frozenset con los GUIDs reales de (pid, comp) en los 4 k-pares.
    El centinela NULL_GUID queda excluido.
    """
    # k=1: ProcessGuid, EID no en {8, 10}
    g1 = (set(df[~df['EventID'].isin([8, 10])
               & (df['Computer'] == comp)
               & (df['ProcessId'] == pid)]['ProcessGuid'].dropna())
          - {NULL_GUID})

    # k=2: ParentProcessGuid, solo EID=1
    # TODO: completar g2
    g2 = set()

    # k=3: SourceProcessGUID, EID en {8, 10}
    # TODO: completar g3
    g3 = set()

    # k=4: TargetProcessGUID, EID en {8, 10}
    # TODO: completar g4
    g4 = set()

    return frozenset(g1 | g2 | g3 | g4)


```{admonition} Pista — estructura de cada k-par
:class: hint

Cada bloque sigue exactamente el mismo patrón que g1:

    gK = (set(df[ <filtro_EID> & (df['Computer']==comp) & (df['<PidCol>']==pid) ]
               ['<GuidCol>'].dropna())
          - {NULL_GUID})

Solo cambian `<filtro_EID>`, `<PidCol>` y `<GuidCol>` — consulta la tabla de arriba.
```


In [ ]:
# Validación — aplica compute_gset a todos los PIDs violadores de k=1
# Resultado esperado: |G|=0 → 1 PID, |G|=1 → 37 PIDs, |G|>1 → 13 PIDs

from collections import Counter

gsize_k1 = Counter()
for (comp, pid), _ in k1_sent.groupby(['Computer', 'ProcessId']):
    g = len(compute_gset(df, pid, comp))
    gsize_k1[0 if g==0 else (1 if g==1 else 'gt1')] += 1

print("Distribución |G| para k=1:")
print(f"  |G|=0 (BOOT_ARTIFACT): {gsize_k1[0]:3d}  ← esperado: 1")
print(f"  |G|=1 (REPLACE_GUID):  {gsize_k1[1]:3d}  ← esperado: 37")
print(f"  |G|>1 (REVIEW):        {gsize_k1['gt1']:3d}  ← esperado: 13")


---
## Actividad 3 — Clasificar las violaciones k=1

Construye la tabla de clasificación completa para los 51 PIDs violadores k=1.
Para cada (comp, pid): calcula |G|, determina la acción y guarda el resultado.


In [ ]:
rows = []
for (comp, pid), grp in k1_sent.groupby(['Computer', 'ProcessId']):
    G = compute_gset(df, pid, comp)
    card = len(G)
    accion = 'REPLACE_GUID' if card == 1 else ('BOOT_ARTIFACT' if card == 0 else 'REVIEW')
    eids   = sorted(grp['EventID'].unique().tolist())
    rows.append({
        'comp':   comp.split('.')[0],
        'pid':    int(pid),
        'n_sent': len(grp),
        'G_size': card,
        'accion': accion,
        'eids':   eids,
    })

tabla = pd.DataFrame(rows).sort_values(['accion', 'comp', 'pid'])
print(tabla.to_string(index=False))
print(f"\nTotal: {len(tabla)} PIDs — "
      + "  ".join(f"{v}: {(tabla['accion']==v).sum()}" for v in ['REPLACE_GUID','REVIEW','BOOT_ARTIFACT']))


```{admonition} Preguntas
:class: tip
1. ¿Cuál es el PID con `BOOT_ARTIFACT`? ¿Qué proceso es? ¿Por qué no tiene ningún GUID real?
2. ¿Qué diferencia hay entre un `BOOT_ARTIFACT` y un `PARENT_PREDATES_SYSMON`?
```


---
## Actividad 4 — Verificación temporal para un caso REPLACE_GUID

Elige un PID con `REPLACE_GUID` (|G|=1) de la tabla anterior y aplica el
protocolo de verificación temporal: determina t*, t_min(g₀), t_max(g₀)
y clasifica el escenario en {PRE_GUID_INIT, WITHIN_WINDOW, POST_GUID_TERMINATE}.


In [ ]:
# Elige un PID de la tabla anterior con accion=='REPLACE_GUID' y reemplaza:
comp_elegido = 'TODO.boombox.local'   # ej. 'diskjockey.boombox.local'
pid_elegido  = 0                       # ej. 1234

if comp_elegido.startswith('TODO') or pid_elegido == 0:
    print("Reemplaza comp_elegido y pid_elegido con un caso REPLACE_GUID de la tabla.")
else:
    # centinelas del caso
    sents = k1_sent[(k1_sent['Computer'] == comp_elegido) &
                    (k1_sent['ProcessId'] == pid_elegido)].sort_values('ts')
    print(f"Centinelas (n={len(sents)}):")
    print(sents[['ts','EventID','Image']].to_string(index=False))

    # G(p, c)
    G = compute_gset(df, pid_elegido, comp_elegido)
    g0 = list(G)[0]
    print(f"\ng0 = {g0}")

    # ventana temporal de g0 en todas las columnas GUID
    mask_g0 = (
        ((df['ProcessGuid']       == g0) & ~df['EventID'].isin([8, 10])) |
        ((df['ParentProcessGuid'] == g0) &  (df['EventID'] == 1))        |
        ((df['SourceProcessGUID'] == g0) &   df['EventID'].isin([8, 10])) |
        ((df['TargetProcessGUID'] == g0) &   df['EventID'].isin([8, 10]))
    ) & (df['Computer'] == comp_elegido)
    refs   = df[mask_g0]['ts'].dropna()
    t_min  = refs.min(); t_max = refs.max()
    t_star = sents['ts'].min()

    print(f"\nt_min(g0) = {t_min}")
    print(f"t*         = {t_star}   <- centinela")
    print(f"t_max(g0) = {t_max}")

    gap_init = (t_star - t_min).total_seconds() * 1000
    gap_term = (t_star - t_max).total_seconds() * 1000
    print(f"\ngap(t* - t_min) = {gap_init:+.1f} ms")
    print(f"gap(t* - t_max) = {gap_term:+.1f} ms")

    if gap_init < 0:
        escenario = 'PRE_GUID_INIT'
    elif gap_term > 0:
        escenario = 'POST_GUID_TERMINATE'
    else:
        escenario = 'WITHIN_WINDOW'
    print(f"\nEscenario: {escenario}")


---
## Actividad 5 — Nuevos patrones: k=3

En `run-01-apt-1` el k=3 tenía **0 violaciones**. En `run-02` aparecen algunas.
Analiza qué proceso genera estas violaciones y por qué tiene `SourceProcessGUID = ∅`.


In [ ]:
print("Violaciones k=3 en run-02:")
for (comp, pid), grp in k3_sent.groupby(['Computer', 'SourceProcessId']):
    G = compute_gset(df, pid, comp)
    short = comp.split('.')[0]
    eids  = sorted(grp['EventID'].unique().tolist())
    # imagen del proceso fuente (SourceImage en EID=10)
    imgs  = grp['SourceImage'].dropna().unique()
    print(f"  PID {int(pid):5d} @ {short:<12} n={len(grp):2d} EIDs={eids}  |G|={len(G)}")
    for img in imgs:
        print(f"    SourceImage = {img}")


```{admonition} Preguntas
:class: tip
1. ¿Qué EventID tienen estos centinelas k=3? ¿Por qué solo puede ser ese EventID?
2. ¿Tienen |G|=1 o |G|>1? ¿Qué acción corresponde?
3. El campo relevante aquí es `SourceProcessGUID`, no `ProcessGuid`.  
   ¿Qué representa físicamente: el proceso fuente de qué tipo de evento?
```


---
## Actividad 6 — Resumen y comparación con run-01

Construye la tabla de resumen completa para `run-02-apt-1` (todos los k-pares)
y compárala con la de `run-01`.


In [ ]:
# Tabla de resumen: (k, n_centinelas, n_pids, n_replace, n_review, n_boot)
sentinels_by_k = {
    1: (k1_sent, 'ProcessId'),
    2: (k2_sent, 'ParentProcessId'),
    3: (k3_sent, 'SourceProcessId'),
    4: (k4_sent, 'TargetProcessId'),
}

print(f"{'k':>3} {'centinelas':>12} {'PIDs':>5} {'REPLACE':>8} {'REVIEW':>7} {'BOOT':>5}")
print('-' * 50)
total_c = total_p = total_r = total_rv = total_b = 0
for k, (sent, pid_col) in sentinels_by_k.items():
    r = rv = b = 0
    pids = sent.groupby(['Computer', pid_col])
    for (comp, pid), _ in pids:
        G = compute_gset(df, pid, comp)
        if len(G) == 1:   r  += 1
        elif len(G) > 1:  rv += 1
        else:             b  += 1
    n_c, n_p = len(sent), sent[pid_col].nunique()
    print(f"{k:>3} {n_c:>12,} {n_p:>5} {r:>8} {rv:>7} {b:>5}")
    total_c+=n_c; total_p+=n_p; total_r+=r; total_rv+=rv; total_b+=b
print('-' * 50)
print(f"{'∑':>3} {total_c:>12,} {'—':>5} {total_r:>8} {total_rv:>7} {total_b:>5}")


```{admonition} Reflexión final
:class: important
Compara tu tabla con la de `run-01-apt-1`. Identifica:

1. ¿Qué mecanismo es dominante en k=2 de run-01 pero casi inexistente en run-02?
2. ¿Por qué aparece k=3 en run-02 pero no en run-01? ¿Qué tipo de actividad APT
   genera eventos EID=10/EID=8 con `SourceProcessGUID = ∅`?
3. El caso `BOOT_ARTIFACT` (|G|=0) no se puede corregir automáticamente.
   ¿Qué harías con estos eventos en el pipeline de ML? ¿Descartarlos? ¿Etiquetarlos?
```
